In [1]:
import numpy as np
import pandas as pd
from scipy import constants
import glob 
import matlab.engine
import matplotlib.pyplot as plt

In [2]:
def svd_beamformer_analog(H):
    ant_tx, ant_rx = np.shape(H)
    u, s, wr_opt = np.linalg.svd(H)
    u, s, wt_opt = np.linalg.svd(np.transpose(H))
    
    wr_quant_angle = -np.transpose(np.around(np.angle(wr_opt)/(np.pi/2))*(np.pi/2))
    wt_quant_angle = -np.transpose(np.around(np.angle(wt_opt)/(np.pi/2))*(np.pi/2))
    
    wr_quant = np.exp(1j*wr_quant_angle)
    wt_quant = np.exp(1j*wt_quant_angle)
    
    rss = []
    beam_idx = []
    for i in range(ant_tx):
        for j in range(ant_rx):
            signal = np.abs(np.matmul(wt_quant[:,i],np.matmul(H,wr_quant[:,j])))**2
            rss.append(10*np.log10(signal*1000))
            beam_idx.append([i,j])
    
    idx = np.argmax(np.array(rss))
    tx_idx = beam_idx[idx][0]
    rx_idx = beam_idx[idx][1]
            
    wr_quant_out = wr_quant[:,rx_idx]
    wt_quant_out = wt_quant[:,tx_idx] 
    
    return wt_quant_out, wr_quant_out

In [3]:
eng = matlab.engine.start_matlab()

In [4]:
azi_files = glob.glob('./output/az*')

In [5]:
tx = {'freq':11.325e9,
      'light_speed':constants.c,
      'power': 4.89e6,
      'num_ant': 1,
      'rf_chain':1,
      'ant_spacing':constants.c/11.325e9/2}

rx = {'freq':11.325e9,
      'light_speed':constants.c,
      'power': 1,
      'num_ant': 36,
      'rf_chain':1,
      'ant_spacing':constants.c/11.325e9/2}

In [6]:
noise = -80

In [11]:
def kernel_function(azi_file, lock = 1):
    print("Now Processing " + azi_file)
    alt_file = azi_file.replace('az','alt')
    dist_file = azi_file.replace('az','distance')
    
    azi_data = pd.read_csv(azi_file)
    alt_data = pd.read_csv(alt_file)
    dist_data = pd.read_csv(dist_file)
    
    rss_ori = np.ones((len(azi_data),1))*np.nan
    rss_beamformed = np.ones((len(azi_data),1))*np.nan
    rate_ori = np.ones((len(azi_data),1))*np.nan
    rate_beamformed = np.ones((len(azi_data),1))*np.nan
    for i in range(len(azi_data)):
        txang_azi = azi_data.iloc[i,2:].values
        txang_azi = txang_azi[txang_azi!=0]-np.pi
        txang_ele = alt_data.iloc[i,2:].values
        txang_ele = txang_ele[txang_ele!=0]
        txang = matlab.double(np.array([txang_azi,txang_ele]).tolist())
       
        rxang_azi = azi_data.iloc[i,2:].values
        rxang_azi = rxang_azi[rxang_azi!=0]-np.pi
        rxang_ele = alt_data.iloc[i,2:].values
        rxang_ele = rxang_ele[rxang_ele!=0]
        rxang = matlab.double(np.array([rxang_azi,rxang_ele]).tolist())
        
        dist = np.array(dist_data.iloc[i,2:].values)*1000
        dist = matlab.double(dist[dist!=0].tolist())
        
        H = np.array(eng.get_channel_MultiTX_SIMO(matlab.double(tx["num_ant"]), matlab.double(rx["num_ant"]), 
                            matlab.double(tx["rf_chain"]), matlab.double(rx["rf_chain"]),
                           matlab.double(tx["freq"]), txang, rxang, dist))
        
        wt, wr = svd_beamformer_analog(H)
        rss_ori_sat = 10*np.log10(np.abs(np.sum(H))**2*tx['power']/tx['num_ant']*1000)
        rss_beamformed_sat = np.squeeze(10*np.log10(np.abs(np.matmul(np.expand_dims(wt, axis = 0), np.matmul(H, np.expand_dims(wr, axis = 1))))**2*tx['power']/tx['num_ant']*1000))
        
        rss_ori[i] = rss_ori_sat
        rss_beamformed[i] = rss_beamformed_sat
        
        rate_ori[i] = np.log2(1 + 10**((rss_ori_sat - noise)/10))
        print(rss_ori_sat - noise)
        rate_beamformed[i] = np.log2(1 + 10**((rss_beamformed_sat - noise)/10))
    
        if np.mod(i+1, 100) == 0 or i+1 == len(azi_data):
            print("Processed " + azi_file + " " + str(i+1) + "/" + str(len(azi_data)) + " time")
    
    fout = azi_file.replace('az','rss_ori_multiTX')
    df_out = pd.DataFrame(rss_ori.tolist(), columns = ["RSS"])
    df_out.to_csv(fout)

    fout = azi_file.replace('az','rss_beamformed_multiTX')
    df_out = pd.DataFrame(rss_beamformed.tolist(), columns = ["RSS_BEAMFORMED"])
    df_out.to_csv(fout)
    
    fout = azi_file.replace('az','rate_ori_multiTX')
    df_out = pd.DataFrame(rate_ori.tolist(), columns = ["RATE"])
    df_out.to_csv(fout)
    
    fout = azi_file.replace('az','rate_beamformed_multiTX')
    df_out = pd.DataFrame(rate_beamformed.tolist(), columns = ["RATE_BEAMFORMED"])
    df_out.to_csv(fout)

In [12]:
for azi_file in azi_files:
    kernel_function(azi_file, 2)

Now Processing ./output\az_Boston_2022-9-21-00-00-00.csv
19.70042112997973
19.553402505985588
19.377675825381424
19.170983442758462
18.28884564893726
18.04405449227979
17.759222668489762
17.430549374484002
17.05326746469062
16.6216636996936
16.12867256883247
15.565399566658925
16.335494033909043
15.691467516611809
14.956270049871293
14.111547087486102
13.131890370257793
11.980595443187482
10.602097549366704
14.200988927121799
13.125357874258512
11.863714976916185
10.351872686965208
16.398012065783433
15.484025125355572
14.447352567320635
10.492808190267013
8.725176085408975
6.486295760204982
9.348560844271049
7.475126977833511
5.077378362420035
1.7519845375205847
-3.6986833101452277
-21.50877095204011
-6.3129024997297165
0.4143851178180711
4.146310294604731
6.736996374960299
13.984471028013104
14.87104878663898
18.73032800542164
19.205273298265382
19.650553061982137
17.413247622107505
17.14748206599249
16.87721919749496
16.60311791340795
16.326184039513564
16.047567364621777
15.7689013

ValueError: not enough values to unpack (expected 2, got 0)